## Extraction of Data from Chirps Dataset (GEE) ##

In [1]:
# Import Google Earth Engine API and Initialize it. 
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project="ey-data-and-ai-challenge")

In [2]:
# Read coordinates and date from water quality training dataset, drop given features.

wq_df = pd.read_csv('../data/water_quality_training_dataset.csv')
wq_df = wq_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
wq_df['id'] = wq_df.index
wq_df['Sample Date'] = pd.to_datetime(wq_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
wq_df.head()

,Latitude,Longitude,Sample Date,id
0,-28.760833,17.730278,2011-01-02,0
1,-26.861111,28.884722,2011-01-03,1
2,-26.450000,28.085833,2011-01-03,2
3,-27.671111,27.236944,2011-01-03,3
4,-27.356667,27.286389,2011-01-03,4


In [9]:
# Convert Coordinates and given date to ee.Features for use in batch export.

features = []

for index, row in wq_df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(5500), #add a 5.5km buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=1)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=1)).strftime('%Y-%m-%d')
        }
    )
    features.append(feat)

fc = ee.FeatureCollection(features)         # create feature collection with features

In [10]:
chirps_collection = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select('precipitation')

In [14]:
def extract_median_values(feat):
    collection = chirps_collection.filterDate(feat.get('start_date'), feat.get('end_date'))
    img = collection.reduce(ee.Reducer.sum()) # reduce image collection into a single image

    chirps_col = img.reduceRegions(collection=ee.FeatureCollection([feat]), reducer=ee.Reducer.first(), scale = 5566)
    
    return chirps_col.first()

In [15]:
fc_mapped = fc.map(extract_median_values)

In [16]:
# Process data and export to Google Drive

task = ee.batch.Export.table.toDrive(
    collection=fc_mapped,
    description="chirps_csv_export",
    fileNamePrefix= "chirps_features_training",
    fileFormat='CSV'
)
task.start()

In [17]:
chirps_df = pd.read_csv("../data/chirps_features_training.csv")

# Drop irrelevant columns
chirps_df.drop(columns=[".geo", "system:index", "end_date", "start_date"], inplace=True)

chirps_df = chirps_df.merge(wq_df, on='id', how='left')
chirps_df.drop(columns=['id'], inplace=True)
chirps_df

,first,Latitude,Longitude,Sample Date
0,0.488024,-28.760833,17.730278,2011-01-02
1,97.342756,-26.861111,28.884722,2011-01-03
2,96.911494,-26.450000,28.085833,2011-01-03
3,102.307634,-27.671111,27.236944,2011-01-03
4,96.513882,-27.356667,27.286389,2011-01-03
...,...,...,...,...
9314,21.902490,-27.527500,30.858056,2015-12-23
9315,25.616568,-26.861111,28.884722,2015-12-23
9316,30.374333,-26.984722,26.632278,2015-12-23
9317,12.569091,-27.935000,26.126667,2015-12-23


In [18]:
chirps_df = chirps_df.rename(columns={'first':'precipitation'})
chirps_df

,precipitation,Latitude,Longitude,Sample Date
0,0.488024,-28.760833,17.730278,2011-01-02
1,97.342756,-26.861111,28.884722,2011-01-03
2,96.911494,-26.450000,28.085833,2011-01-03
3,102.307634,-27.671111,27.236944,2011-01-03
4,96.513882,-27.356667,27.286389,2011-01-03
...,...,...,...,...
9314,21.902490,-27.527500,30.858056,2015-12-23
9315,25.616568,-26.861111,28.884722,2015-12-23
9316,30.374333,-26.984722,26.632278,2015-12-23
9317,12.569091,-27.935000,26.126667,2015-12-23


In [19]:
chirps_df.to_csv("../data/chirps_features_training.csv")        # now ready to go for preprocessing

# Repeat for Validation Set

In [20]:
val_df = pd.read_csv("../data/submission_template.csv")
val_df['id'] = val_df.index
val_df['Sample Date'] = pd.to_datetime(val_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
val_df

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,id
0,-32.043333,27.822778,2014-09-01,NaN,NaN,NaN,0
1,-33.329167,26.077500,2015-09-16,NaN,NaN,NaN,1
2,-32.991639,27.640028,2015-05-07,NaN,NaN,NaN,2
3,-34.096389,24.439167,2012-02-07,NaN,NaN,NaN,3
4,-32.000556,28.581667,2014-10-01,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...
195,-33.771111,25.386667,2012-12-06,NaN,NaN,NaN,195
196,-33.185361,27.390750,2014-09-04,NaN,NaN,NaN,196
197,-32.043333,27.822778,2015-09-28,NaN,NaN,NaN,197
198,-33.001667,25.161389,2015-01-08,NaN,NaN,NaN,198


In [21]:
# Convert Coordinates and given date to ee.Features for use in batch export.

features_val = []

for index, row in val_df.iterrows():
    feat_val = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(5500), #add a 5.5km buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=1)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=1)).strftime('%Y-%m-%d')
        }
    )
    features_val.append(feat_val)

fc_val = ee.FeatureCollection(features_val)         # create feature collection with features

In [22]:
fc_mapped_val = fc_val.map(extract_median_values)

In [23]:
task = ee.batch.Export.table.toDrive(
    collection=fc_mapped_val,
    description="chirps_val_csv_export",
    fileNamePrefix= "chirps_features_validation",
    fileFormat='CSV'
)
task.start()

In [24]:
chirps_val_df = pd.read_csv("../data/chirps_features_validation.csv")

# Drop irrelevant columns
chirps_val_df.drop(columns=[".geo", "system:index", "end_date", "start_date"], inplace=True)
chirps_val_df = chirps_val_df.rename(columns={'first':'precipitation'})

chirps_val_df = chirps_val_df.merge(wq_df, on='id', how='left')
chirps_val_df.drop(columns=['id'], inplace=True)
chirps_val_df

,precipitation,Latitude,Longitude,Sample Date
0,0.000000,-28.760833,17.730278,2011-01-02
1,27.521283,-26.861111,28.884722,2011-01-03
2,14.464016,-26.450000,28.085833,2011-01-03
3,42.085633,-27.671111,27.236944,2011-01-03
4,30.467072,-27.356667,27.286389,2011-01-03
...,...,...,...,...
195,12.736896,-33.818056,19.694722,2011-02-22
196,0.000000,-25.810483,27.909552,2011-02-23
197,9.080177,-29.641944,30.687500,2011-02-23
198,11.771285,-34.065833,20.404167,2011-02-23


In [25]:
chirps_val_df.to_csv("../data/chirps_features_validation.csv")